In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np

c:\Users\konta\anaconda3\envs\tk\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
dataset = load_dataset('amazon_polarity')
df = pd.DataFrame(dataset['train'])[['content', 'label']]
df = df.sample(200000, random_state=123)

In [ ]:
df_test = pd.DataFrame(dataset['test'])[['content', 'label']]
df_test = df.sample(50000, random_state=123)

In [ ]:
df['content']

In [ ]:
pd.Series(df['label']).value_counts()

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from collections import Counter
import re

MAX_VOCAB = 20000
MAX_LEN = 200
BATCH_SIZE = 64
EMBED_DIM = 128
HIDDEN_DIM = 128
EPOCHS = 3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset = load_dataset("amazon_polarity")

train_data = dataset["train"]
test_data = dataset["test"]

train_texts = train_data["content"]
test_texts = test_data["content"]

train_labels = train_data["label"]
test_labels = test_data["label"]

print("Train size:", len(train_texts))
print("Test size:", len(test_texts))


def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\\s]", "", text)
    return text


def tokenize(text):
    return clean_text(text).split()


counter = Counter()

for text in train_texts[:100000]:   
    counter.update(tokenize(text))

most_common = counter.most_common(MAX_VOCAB - 2)

vocab = {
    "<PAD>": 0,
    "<UNK>": 1
}

for idx, (word, _) in enumerate(most_common, start=2):
    vocab[word] = idx


def encode(text):
    tokens = tokenize(text)
    seq = [vocab.get(token, vocab["<UNK>"]) for token in tokens]

    if len(seq) < MAX_LEN:
        seq += [vocab["<PAD>"]] * (MAX_LEN - len(seq))
    else:
        seq = seq[:MAX_LEN]

    return seq


class AmazonDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        x = torch.tensor(encode(self.texts[idx]), dtype=torch.long)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y


train_dataset = AmazonDataset(train_texts, train_labels)
test_dataset = AmazonDataset(test_texts, test_labels)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)


class SentimentLSTM(nn.Module):
    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(MAX_VOCAB, EMBED_DIM, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size=EMBED_DIM,
            hidden_size=HIDDEN_DIM,
            batch_first=True
        )
        self.fc1 = nn.Linear(HIDDEN_DIM, 64)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(64, 2)   

        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.embedding(x)
        _, (hidden, _) = self.lstm(x)

        x = hidden[-1]
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)

        return x


model = SentimentLSTM().to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss/len(train_loader):.4f}")


model.eval()
correct = 0
total = 0

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        outputs = model(X_batch)
        predictions = torch.argmax(outputs, dim=1)

        correct += (predictions == y_batch).sum().item()
        total += y_batch.size(0)

accuracy = correct / total
print("Test Accuracy:", accuracy)

def predict_sentiment(text):
    model.eval()

    encoded = torch.tensor([encode(text)], dtype=torch.long).to(DEVICE)

    with torch.no_grad():
        output = model(encoded)
        pred = torch.argmax(output, dim=1).item()

    return "Positive" if pred == 1 else "Negative"


samples = [
    "I absolutely love this product, works perfectly!",
    "Worst purchase ever, very disappointed."
]

for text in samples:
    print(text, "->", predict_sentiment(text))

c:\Users\konta\anaconda3\envs\tk\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\konta\anaconda3\envs\tk\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\konta\.cache\huggingface\hub\datasets--amazon_polarity. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode,

Train size: 3600000
Test size: 400000


KeyboardInterrupt: 